# V12 Phase 4 2x2 — matched vision-only vs joint (Flickr8k)

4 arms × 500 steps. A=vis/ATTN, B=vis/V8-A, C=joint/ATTN, D=joint/V8-A.
Same data, init (tensor-verified), optimizer. Vision loss identical across arms.
I = Δ_joint − Δ_vis-only; I<0 → rescue.


In [ ]:
import os
import torch
assert torch.cuda.is_available(), "needs GPU"
device = "cuda"
def set_seed(s):
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(0)
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))


In [ ]:
import urllib.request, zipfile, glob
from PIL import Image
import re
from collections import Counter

def dl(url, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        print("downloading", url.split("/")[-1], flush=True)
        urllib.request.urlretrieve(url, path)
    return path

img_zip = dl("https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip",
             "data/Flickr8k_Dataset.zip")
txt_zip = dl("https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip",
             "data/Flickr8k_text.zip")
IMG_DIR = "data/Flicker8k_Dataset"
if not os.path.exists(IMG_DIR):
    print("extracting images...", flush=True)
    with zipfile.ZipFile(img_zip) as z:
        z.extractall("data")
_img_dirs = glob.glob("data/**/Flicker8k_Dataset", recursive=True)
if _img_dirs:
    IMG_DIR = _img_dirs[0]
print(f"image dir: {IMG_DIR}")

TXT_DIR = "data/Flickr8k_text"
if not os.path.exists(TXT_DIR):
    print("extracting text...", flush=True)
    with zipfile.ZipFile(txt_zip) as z:
        z.extractall("data")
_tok_paths = glob.glob("data/**/Flickr8k.token.txt", recursive=True)
assert _tok_paths, "Flickr8k.token.txt not found"
TOKEN_PATH = _tok_paths[0]
print(f"token file: {TOKEN_PATH}")

pairs = {}
with open(TOKEN_PATH) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        img_cap, caption = line.split("\t")
        img_name = img_cap.split("#")[0]
        pairs.setdefault(img_name, []).append(caption)
items = [(n, cs[0]) for n, cs in pairs.items()
         if os.path.exists(f"{IMG_DIR}/{n}")]
print(f"paired items: {len(items)}")

def tok(s):
    return re.findall(r"[a-z]+", s.lower())
cnt = Counter()
for _, c in items:
    cnt.update(tok(c))
vocab = {"<pad>": 0, "<unk>": 1}
for w, _ in cnt.most_common(8000):
    vocab[w] = len(vocab)
VOCAB = len(vocab)
print(f"vocab: {VOCAB}")

def encode_caption(s, T):
    ids = [vocab.get(w, 1) for w in tok(s)][:T]
    ids += [0] * (T - len(ids))
    return ids

def load_image(name):
    return Image.open(f"{IMG_DIR}/{name}").convert("RGB")


In [ ]:
"""
Linear-recurrent replacement for Transformer self-attention.

Experiment: take a standard decoder-only Transformer, rip out the
causal self-attention layer, and replace it with a *moving linear
equation* that compresses the past into a fixed-size running state:

    h_t = d * h_{t-1} + (1 - d) * (B x_t)        (linear recurrence)
    y_t = C h_t                                   (readout)

where d in (0,1)^S is a learned per-channel decay, B/C are learned
projections, and h has FIXED size S independent of sequence length.

This is O(T) time and O(1) memory w.r.t. context length, vs O(T^2)
for attention. The question: how much modeling power do we lose?
"""

import math
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------
# Token mixing layers
# ----------------------------------------------------------------------------

class SelectiveSegmentedState(nn.Module):
    """V5: gated segmented state-space mixer.

    Replaces the static decay with two dynamic mechanisms:

    1. Dynamic Selective Gate (the model's "will"):
         g_t = sigmoid(W_g x_t + b_g)      in (0,1), per token & channel
       acts as a learned per-token filter: low-information tokens get
       g_t -> 0 (forgotten fast), salient tokens get g_t -> 1 (kept).

    2. Hard Memory Reset Mask (segment boundaries):
         r_t = 1 if token t is a structural boundary (e.g. newline)
         h_t = (1 - r_t) * (g_t * h_{t-1} + (1 - g_t) * u_t)
       when r_t = 1 the state is multiplied by zero: the past memory
       buffer is completely destroyed and the new segment starts fresh.
       (The residual stream still carries x_t itself forward.)

    The recurrence stays affine in h (h_t = a_t*h_{t-1} + b_t with
    a_t = (1-r_t)*g_t, b_t = (1-r_t)*(1-g_t)*u_t), so the parallel
    associative scan still applies: O(T) work, log2(T) parallel passes.
    State dim stays frozen at `state_dim` (VRAM edge preserved).
    """
    def __init__(self, dim, state_dim=256, dropout=0.0):
        super().__init__()
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)  # bias = per-channel prior
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        # x: (B,T,dim); reset: (B,T) binary, 1 = boundary token
        u = self.in_proj(x)                    # (B,T,S)
        g = torch.sigmoid(self.gate_proj(x))   # (B,T,S) dynamic gate
        if reset is not None:
            keep = (1.0 - reset).unsqueeze(-1)  # (B,T,1)
            a = keep * g
            b = keep * (1.0 - g) * u
        else:
            a = g
            b = (1.0 - g) * u
        h_seq = parallel_scan_ab(a, b)          # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


class CausalSelfAttention(nn.Module):
    """Standard multi-head causal self-attention (the baseline)."""
    def __init__(self, dim, n_heads, dropout=0.0):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)                       # (B,T,H,D)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))  # (B,H,T,D)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(y)


def parallel_scan_ab(A, B):
    """Parallel associative scan (Hillis-Steele) for h_t = A_t*h_{t-1} + B_t.

    A, B: (B, T, S) per-step coefficients. Returns H: (B, T, S).
    log2(T) fully-parallel passes instead of T sequential steps.
    """
    Bsz, T, S = A.shape
    A = A.contiguous()
    Bh = B.contiguous()
    idx = torch.arange(T, device=A.device).view(1, T, 1)
    o = 1
    while o < T:
        m = idx >= o
        A_sh = torch.roll(A, o, dims=1)
        B_sh = torch.roll(Bh, o, dims=1)
        # compose segment [t-o, t-1] then [t]:
        #   a_new = a_t * a_{t-o},  b_new = a_t * b_{t-o} + b_t
        A_new = A * A_sh
        B_new = A * B_sh + Bh
        A = torch.where(m, A_new, A)
        Bh = torch.where(m, B_new, Bh)
        o *= 2
    return Bh


def parallel_linear_recurrence(u, decay):
    """Parallel associative scan (Hillis-Steele) for h_t = d*h_{t-1} + c_t.

    Reformulates the sequential loop as a tree reduction:
      - each step is an affine map  h -> a*h + b, packed as pair (a, b)
      - pair composition is associative:
            (a1,b1) then (a2,b2)  =  (a2*a1, a2*b1 + b2)
      - a parallel prefix over pairs yields every h_t at once

    log2(T) fully-parallel passes instead of T sequential steps.
    u: (B, T, S) inputs already projected; decay: (S,) in (0, 1).
    Returns h: (B, T, S) — identical math to the for-loop version.
    """
    Bsz, T, S = u.shape
    c = (1.0 - decay) * u
    A = decay.expand(Bsz, T, S).contiguous()   # a-coefficients
    return parallel_scan_ab(A, c)


class LinearRunningState(nn.Module):
    """Attention replacement: fixed-size running state, linear recurrence.

    h_t = decay * h_{t-1} + (1 - decay) * (B x_t)
    y_t = C h_t

    - decay: learned per-state-channel forget factor in (0, 1)
    - B:     input  projection  dim -> state_dim
    - C:     output projection  state_dim -> dim
    State size is FIXED (does not grow with context length).
    mode="loop": native Python for-loop (sequential, slow on GPU)
    mode="scan": parallel associative scan (log2(T) parallel passes)
    """
    def __init__(self, dim, state_dim=None, dropout=0.0, mode="loop"):
        super().__init__()
        self.mode = mode
        self.state_dim = state_dim or dim
        # init decay ~ U(0.9, 0.999) -> long-ish memory at start
        init = torch.empty(self.state_dim).uniform_(math.log(0.9 / 0.1),
                                                   math.log(0.999 / 0.001))
        self.logit_decay = nn.Parameter(init)
        self.in_proj = nn.Linear(dim, self.state_dim, bias=False)
        self.out_proj = nn.Linear(self.state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        B, T, _ = x.shape
        decay = torch.sigmoid(self.logit_decay)          # (S,) in (0,1)
        u = self.in_proj(x)                              # (B,T,S)
        if self.mode == "scan":
            h_seq = parallel_linear_recurrence(u, decay)  # (B,T,S)
        else:
            keep = 1.0 - decay
            h = torch.zeros(B, self.state_dim, device=x.device, dtype=x.dtype)
            outs = []
            for t in range(T):
                h = decay * h + keep * u[:, t]
                outs.append(h)
            h_seq = torch.stack(outs, dim=1)              # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


# ----------------------------------------------------------------------------
# Blocks & models (identical except for the mixing layer)
# ----------------------------------------------------------------------------

class Block(nn.Module):
    def __init__(self, dim, n_heads, mixer, dropout=0.0, ffn_hidden=None):
        super().__init__()
        ffn_hidden = ffn_hidden or 4 * dim
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.mix = mixer
        self.mlp = nn.Sequential(
            nn.Linear(dim, ffn_hidden), nn.GELU(),
            nn.Linear(ffn_hidden, dim), nn.Dropout(dropout))

    def forward(self, x, reset=None):
        x = x + self.mix(self.ln1(x), reset)
        x = x + self.mlp(self.ln2(x))
        return x


class TinyLM(nn.Module):
    """Decoder-only LM. mixer_fn(dim, n_heads) builds the token-mixing layer."""
    def __init__(self, vocab, dim=128, n_layers=4, n_heads=4,
                 seq_len=128, mixer_fn=None, dropout=0.0, ffn_hidden=None,
                 newline_id=None):
        super().__init__()
        self.seq_len = seq_len
        self.newline_id = newline_id  # boundary token id for reset mask
        self.tok_emb = nn.Embedding(vocab, dim)
        self.pos_emb = nn.Embedding(seq_len, dim)
        self.blocks = nn.ModuleList(
            [Block(dim, n_heads, mixer_fn(dim, n_heads), dropout, ffn_hidden)
             for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(
            torch.arange(T, device=idx.device))[None]
        reset = None
        if self.newline_id is not None:
            reset = (idx == self.newline_id).float()  # (B,T) boundary mask
        for blk in self.blocks:
            x = blk(x, reset)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new, temp=1.0, top_k=None):
        for _ in range(max_new):
            ctx = idx[:, -self.seq_len:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :] / max(temp, 1e-6)
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def build_models(vocab, dim=128, n_layers=4, n_heads=4, seq_len=128,
                 state_dim=None, dropout=0.0):
    attn = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                  mixer_fn=lambda d, h: CausalSelfAttention(d, h, dropout),
                  dropout=dropout)
    lin = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                 mixer_fn=lambda d, h: LinearRunningState(d, state_dim, dropout),
                 dropout=dropout)
    return attn, lin


"""(inlined) V6/V7/V8-A: Triton-fused scans.

Fuses the V5 recurrence (per the V6 execution plan):
  Step A: the 256-wide hidden state lives in SRAM registers for the whole
          sequence -- loaded once per batch element, never spilled to HBM.
  Step B: sigmoid gate, hard reset mask, and the linear update
          h = keep*g*h + keep*(1-g)*u run in registers, every timestep.
  Step C: only h_t (plus a_t, saved for the backward pass) hit HBM.
          The parallel scan's O(log T) intermediate (B,T,S) tensors vanish.

Math is IDENTICAL to V5's SelectiveSegmentedState -- only the execution
path changes. Zero new parameters.
"""
import triton
import triton.language as tl

BLOCK_S = 256  # state dim; the whole state fits in SRAM


@triton.jit
def _sel_scan_fwd_kernel(u_ptr, gp_ptr, keep_ptr, h_ptr, a_ptr,
                         T, BLOCK: tl.constexpr):
    """Forward: h_t = a_t*h_{t-1} + b_t with a_t = keep_t*g_t,
    b_t = keep_t*(1-g_t)*u_t, g_t = sigmoid(gp_t)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    u_b = u_ptr + b * bs
    gp_b = gp_ptr + b * bs
    h_b = h_ptr + b * bs
    a_b = a_ptr + b * bs
    k_b = keep_ptr + b * T
    h = tl.zeros([BLOCK], dtype=tl.float32)          # Step A: state in SRAM
    for t in range(T):
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        gp_t = tl.load(gp_b + t * BLOCK + offs).to(tl.float32)
        k = tl.load(k_b + t).to(tl.float32)
        g = 1.0 / (1.0 + tl.exp(-gp_t))              # Step B: gate in registers
        at = k * g
        h = at * h + k * (1.0 - g) * u_t             # Step B: update in registers
        tl.store(h_b + t * BLOCK + offs, h)          # Step C: outputs to HBM
        tl.store(a_b + t * BLOCK + offs, at)


@triton.jit
def _sel_scan_bwd_kernel(dh_ptr, hs_ptr, ap_ptr, u_ptr, gp_ptr, keep_ptr,
                         du_ptr, dgp_ptr, T, BLOCK: tl.constexpr):
    """Backward. hs is (B,T+1,S) with hs[t] = h_{t-1} (hs[0] = 0);
    ap is (B,T+1,S) with ap[t+1] = a_{t+1} (ap[T] = 0).
    D_t = gout_t + a_{t+1}*D_{t+1}; du_t = k(1-g)D_t;
    dgp_t = k*g*(1-g)*(h_{t-1}-u_t)*D_t."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    bsp = (T + 1) * BLOCK
    dh_b = dh_ptr + b * bs
    hs_b = hs_ptr + b * bsp
    ap_b = ap_ptr + b * bsp
    u_b = u_ptr + b * bs
    gp_b = gp_ptr + b * bs
    k_b = keep_ptr + b * T
    du_b = du_ptr + b * bs
    dgp_b = dgp_ptr + b * bs
    dnext = tl.zeros([BLOCK], dtype=tl.float32)      # D_{t+1}, SRAM-resident
    for ti in range(T):
        t = T - 1 - ti
        gout = tl.load(dh_b + t * BLOCK + offs).to(tl.float32)
        a_next = tl.load(ap_b + (t + 1) * BLOCK + offs).to(tl.float32)
        D = gout + a_next * dnext
        h_prev = tl.load(hs_b + t * BLOCK + offs).to(tl.float32)
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        gp_t = tl.load(gp_b + t * BLOCK + offs).to(tl.float32)
        k = tl.load(k_b + t).to(tl.float32)
        g = 1.0 / (1.0 + tl.exp(-gp_t))
        omg = 1.0 - g
        tl.store(du_b + t * BLOCK + offs, k * omg * D)
        tl.store(dgp_b + t * BLOCK + offs, k * g * omg * (h_prev - u_t) * D)
        dnext = D


class _SelScanFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, gp, keep):
        assert u.is_cuda and u.dtype == torch.float32, "V6 kernel: CUDA fp32"
        assert u.shape == gp.shape and u.dim() == 3
        assert keep.shape == u.shape[:2]
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, gp, keep = u.contiguous(), gp.contiguous(), keep.contiguous()
        h = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _sel_scan_fwd_kernel[(B,)](u, gp, keep, h, a, T,
                                   BLOCK=BLOCK_S, num_warps=4)
        ctx.save_for_backward(u, gp, keep, h, a)
        return h

    @staticmethod
    def backward(ctx, dh):
        u, gp, keep, h, a = ctx.saved_tensors
        B, T, S = u.shape
        dh = dh.contiguous()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h    # hs[t] = h_{t-1}
        ap = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        ap[:, :T] = a    # ap[t+1] = a_{t+1}
        du = torch.empty_like(u)
        dgp = torch.empty_like(gp)
        _sel_scan_bwd_kernel[(B,)](dh, hs, ap, u, gp, keep, du, dgp, T,
                                   BLOCK=BLOCK_S, num_warps=4)
        return du, dgp, None


def triton_selective_scan(u, gp, keep):
    """Fused selective scan. u, gp: (B,T,256) fp32 CUDA; keep: (B,T) fp32.
    Returns h: (B,T,256). Differentiable."""
    return _SelScanFn.apply(u, gp, keep)


class SelectiveSegmentedStateTriton(nn.Module):
    """V6 mixer: IDENTICAL math and parameters to V5's SelectiveSegmentedState.

    The only change is execution: the recurrence runs in the fused Triton
    kernel (Step A/B/C) instead of the PyTorch Hillis-Steele scan.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V6 kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        gp = self.gate_proj(x).float().contiguous()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_selective_scan(u, gp, keep)
        return self.dropout(self.out_proj(h_seq.to(x.dtype)))


class SelectiveSegmentedStateV7(nn.Module):
    """V7 mixer: V6's Triton-fused selective scan + two literature grafts.

    Graft #1 — output gate + state normalization (HGRN2 / Mamba-2 / GDN):
        y_t = C · (o_t ⊙ RMSNorm(h_t)),   o_t = σ(W_o x_t + b_o)
    States accumulate over segments so readout scale drifts; every strong
    post-Mamba block gates the readout. RMSNorm here is parameter-free
    (no scale vector) to keep the parameter budget exact.

    Graft #3 — layer-graded forget floor (HGRN, NeurIPS'23):
        g_t = σ(W_g x_t + b_g + floor_l)
    floor_l is a per-layer learnable scalar, initialized monotonically
    increasing bottom→top, so low layers forget fast (spelling-level) and
    top layers retain (phrase-level). Complements the newline hard reset.

    The recurrence math is UNCHANGED → the same fused Triton kernel as V6.
    With FFN 1023 the budget lands EXACTLY on the attention baseline.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0, forget_floor_init=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V7 kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)
        self.out_gate_proj = nn.Linear(dim, state_dim)
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.forget_floor = nn.Parameter(
            torch.tensor(float(forget_floor_init)))
        self.dropout = nn.Dropout(dropout)
        self.eps = 1e-6

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        gp = (self.gate_proj(x) + self.forget_floor).float().contiguous()
        og = self.out_gate_proj(x).float()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_selective_scan(u, gp, keep)          # (B,T,S)
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


# ----------------------------------------------------------------------------
# V8-A: diagonal gated delta-rule scan
# ----------------------------------------------------------------------------

@triton.jit
def _delta_scan_fwd_kernel(u_ptr, ap_ptr, bp_ptr, kp_ptr, keep_ptr,
                           h_ptr, a_ptr, T, BLOCK: tl.constexpr):
    """Forward: h_t = a_t*h_{t-1} + b_t with
        alpha = sigmoid(ap_t), beta = sigmoid(bp_t), kk = sigmoid(kp_t),
        v = silu(u_t),
        a_t = keep_t * alpha * (1 - beta*kk^2),
        b_t = keep_t * beta * kk * v.
    Diagonal restriction of Gated DeltaNet's S_t = a_t S_{t-1}(I-b_t k k^T)
    + b_t v k^T to per-channel 1x1 systems. Strictly contractive:
    sigmoid bounds keep (1-beta*kk^2) in (0,1)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    u_b = u_ptr + b * bs
    ap_b = ap_ptr + b * bs
    bp_b = bp_ptr + b * bs
    kp_b = kp_ptr + b * bs
    h_b = h_ptr + b * bs
    a_b = a_ptr + b * bs
    k_b = keep_ptr + b * T
    h = tl.zeros([BLOCK], dtype=tl.float32)          # state in SRAM
    for t in range(T):
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        ap_t = tl.load(ap_b + t * BLOCK + offs).to(tl.float32)
        bp_t = tl.load(bp_b + t * BLOCK + offs).to(tl.float32)
        kp_t = tl.load(kp_b + t * BLOCK + offs).to(tl.float32)
        kr = tl.load(k_b + t).to(tl.float32)
        alpha = 1.0 / (1.0 + tl.exp(-ap_t))
        beta = 1.0 / (1.0 + tl.exp(-bp_t))
        kk = 1.0 / (1.0 + tl.exp(-kp_t))
        su = 1.0 / (1.0 + tl.exp(-u_t))
        v = u_t * su                                 # SiLU in registers
        at = kr * alpha * (1.0 - beta * kk * kk)
        bt = kr * beta * kk * v
        h = at * h + bt
        tl.store(h_b + t * BLOCK + offs, h)
        tl.store(a_b + t * BLOCK + offs, at)


@triton.jit
def _delta_scan_bwd_kernel(dh_ptr, hs_ptr, apn_ptr, u_ptr, ap_ptr, bp_ptr,
                           kp_ptr, keep_ptr, du_ptr, dap_ptr, dbp_ptr,
                           dkp_ptr, T, BLOCK: tl.constexpr):
    """Backward. hs is (B,T+1,S) with hs[t] = h_{t-1} (hs[0] = 0);
    apn is (B,T+1,S) with apn[t+1] = a_{t+1} (apn[T] = 0).
    Adjoint: D_t = gout_t + a_{t+1}*D_{t+1}; then per-step
      du  = D*kr*beta*kk*silu'(u)
      dap = D*h_prev*kr*alpha*(1-alpha)*(1-beta*kk^2)
      dbp = D*kr*beta*(1-beta)*(kk*v - alpha*kk^2*h_prev)
      dkp = D*kr*beta*kk*(1-kk)*(v - 2*alpha*kk*h_prev)
    (hand-derived; verified against autograd on CPU before transcription)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    bsp = (T + 1) * BLOCK
    dh_b = dh_ptr + b * bs
    hs_b = hs_ptr + b * bsp
    apn_b = apn_ptr + b * bsp
    u_b = u_ptr + b * bs
    ap_b = ap_ptr + b * bs
    bp_b = bp_ptr + b * bs
    kp_b = kp_ptr + b * bs
    k_b = keep_ptr + b * T
    du_b = du_ptr + b * bs
    dap_b = dap_ptr + b * bs
    dbp_b = dbp_ptr + b * bs
    dkp_b = dkp_ptr + b * bs
    dnext = tl.zeros([BLOCK], dtype=tl.float32)      # D_{t+1}, SRAM-resident
    for ti in range(T):
        t = T - 1 - ti
        gout = tl.load(dh_b + t * BLOCK + offs).to(tl.float32)
        a_next = tl.load(apn_b + (t + 1) * BLOCK + offs).to(tl.float32)
        D = gout + a_next * dnext
        h_prev = tl.load(hs_b + t * BLOCK + offs).to(tl.float32)
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        ap_t = tl.load(ap_b + t * BLOCK + offs).to(tl.float32)
        bp_t = tl.load(bp_b + t * BLOCK + offs).to(tl.float32)
        kp_t = tl.load(kp_b + t * BLOCK + offs).to(tl.float32)
        kr = tl.load(k_b + t).to(tl.float32)
        alpha = 1.0 / (1.0 + tl.exp(-ap_t))
        beta = 1.0 / (1.0 + tl.exp(-bp_t))
        kk = 1.0 / (1.0 + tl.exp(-kp_t))
        su = 1.0 / (1.0 + tl.exp(-u_t))
        v = u_t * su
        silu_p = su * (1.0 + u_t * (1.0 - su))
        k2 = kk * kk
        omb = 1.0 - beta * k2
        tl.store(du_b + t * BLOCK + offs,
                 D * kr * beta * kk * silu_p)
        tl.store(dap_b + t * BLOCK + offs,
                 D * h_prev * kr * alpha * (1.0 - alpha) * omb)
        tl.store(dbp_b + t * BLOCK + offs,
                 D * kr * beta * (1.0 - beta) * (kk * v - alpha * k2 * h_prev))
        tl.store(dkp_b + t * BLOCK + offs,
                 D * kr * beta * kk * (1.0 - kk) * (v - 2.0 * alpha * kk * h_prev))
        dnext = D


def _delta_scan_cpu(u, ap, bp, kp, keep):
    """CPU reference for the delta scan: differentiable pure-PyTorch loop.

    Used ONLY for local (CPU) verification. On CUDA the fused Triton
    kernel above is used instead (gate-checked against this math on Colab).
    """
    B, T, S = u.shape
    kr = keep.unsqueeze(-1)
    alpha = torch.sigmoid(ap)
    beta = torch.sigmoid(bp)
    kk = torch.sigmoid(kp)
    v = u * torch.sigmoid(u)                          # SiLU
    h = torch.zeros(B, S, dtype=u.dtype, device=u.device)
    hs = []
    for t in range(T):
        a = kr[:, t] * alpha[:, t] * (1.0 - beta[:, t] * kk[:, t] ** 2)
        bb = kr[:, t] * beta[:, t] * kk[:, t] * v[:, t]
        h = a * h + bb
        hs.append(h)
    return torch.stack(hs, dim=1)


class _DeltaScanFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, ap, bp, kp, keep):
        assert u.is_cuda and u.dtype == torch.float32, \
            "V8-A kernel: CUDA fp32"
        for t in (u, ap, bp, kp):
            assert t.shape == u.shape and t.dim() == 3
        assert keep.shape == u.shape[:2]
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, ap, bp, kp = u.contiguous(), ap.contiguous(), bp.contiguous(), kp.contiguous()
        keep = keep.contiguous()
        h = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _delta_scan_fwd_kernel[(B,)](u, ap, bp, kp, keep, h, a, T,
                                    BLOCK=BLOCK_S, num_warps=4)
        ctx.save_for_backward(u, ap, bp, kp, keep, h, a)
        return h

    @staticmethod
    def backward(ctx, dh):
        u, ap, bp, kp, keep, h, a = ctx.saved_tensors
        B, T, S = u.shape
        dh = dh.contiguous()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h      # hs[t] = h_{t-1}
        apn = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        apn[:, :T] = a     # apn[t+1] = a_{t+1}
        du = torch.empty_like(u)
        dap = torch.empty_like(ap)
        dbp = torch.empty_like(bp)
        dkp = torch.empty_like(kp)
        _delta_scan_bwd_kernel[(B,)](dh, hs, apn, u, ap, bp, kp, keep,
                                    du, dap, dbp, dkp, T,
                                    BLOCK=BLOCK_S, num_warps=4)
        return du, dap, dbp, dkp, None


def triton_delta_scan(u, ap, bp, kp, keep):
    """Fused diagonal delta-rule scan.

    u: raw Bx (SiLU applied in-kernel); ap/bp/kp: pre-sigmoid
    alpha/beta/key projections; keep: (B,T) reset mask.
    (B,T,256) fp32; CUDA -> Triton kernel, CPU -> differentiable
    reference loop (verification only). Returns h: (B,T,256).
    """
    if u.is_cuda:
        return _DeltaScanFn.apply(u, ap, bp, kp, keep)
    return _delta_scan_cpu(u, ap, bp, kp, keep)


class SelectiveSegmentedStateV8A(nn.Module):
    """V8-A mixer: diagonal gated delta rule on V7's chassis.

    Per channel (delta rule = erase-before-write, Gated DeltaNet Eq.10
    restricted to 1x1 systems):
        alpha_t = sigmoid(W_a x_t + b_a + floor_l)   # reuse gate_proj+floor
        beta_t  = sigmoid(W_b x_t + b_b),  b_b = 0   # NEW write_proj
        k_t     = sigmoid(W_k x_t + b_k),  b_k = +2  # NEW key_proj
        v_t     = SiLU(B x_t)                        # reuse in_proj + SiLU
        a_t = alpha_t * (1 - beta_t * k_t^2)
        b_t = beta_t * k_t * v_t
        h_t = (1 - r_t) * (a_t .* h_{t-1} + b_t)
        y_t = C . (o_t .* RMSNorm(h_t)), o_t = sigmoid(W_o x_t)

    Init puts the model near V7's convex-blend regime (k~=0.88, beta~=0.5)
    and the sigmoid bounds keep every step strictly contractive.
    With FFN 766 the budget lands within the relaxed <=3000 gate.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0, forget_floor_init=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V8-A kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)   # B
        self.gate_proj = nn.Linear(dim, state_dim)             # W_alpha
        self.write_proj = nn.Linear(dim, state_dim)            # W_beta (NEW)
        self.key_proj = nn.Linear(dim, state_dim)              # W_k (NEW)
        self.out_gate_proj = nn.Linear(dim, state_dim)         # W_o
        self.out_proj = nn.Linear(state_dim, dim, bias=False)  # C
        self.forget_floor = nn.Parameter(
            torch.tensor(float(forget_floor_init)))
        self.write_proj.bias.data.zero_()       # b_beta = 0  -> beta ~= 0.5
        self.key_proj.bias.data.fill_(2.0)       # b_k = +2.0  -> k ~= 0.88
        self.dropout = nn.Dropout(dropout)
        self.eps = 1e-6
        self.last_state = None  # stashed h_seq for stability logging (no grad)

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        ap = (self.gate_proj(x) + self.forget_floor).float().contiguous()
        bp = self.write_proj(x).float().contiguous()
        kp = self.key_proj(x).float().contiguous()
        og = self.out_gate_proj(x).float()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_delta_scan(u, ap, bp, kp, keep)      # (B,T,S)
        self.last_state = h_seq.detach()  # stability logging only; no numerics change
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


"""(inlined) V11-D1 Path B: fused projection+scan.

Replaces the SAVED projection tensors (u/ap/bp/kp, ~134-400 MB live) with
recomputation. Forward runs the 5 projections with cuBLAS (transient, freed
immediately, never saved) and the EXISTING Triton scan kernels; saves only
(x, keep, h_seq, a_seq). Backward recomputes the projections and runs the
EXISTING verified Triton bwd kernel, then standard dx/dW.

CPU/GPU dispatch: on CUDA the Triton kernels run; on CPU a pure-torch
adjoint loop runs (local verification only — validates the Function's
backward math against torch autograd of the reference).
"""
# (imports stripped)


def _adjoint_torch(dh, x, keep, h_seq, a_seq,
                   w_in, w_a, b_a, floor, w_b, b_b, w_k, b_k):
    """CPU-only adjoint: mirrors _delta_scan_bwd_kernel's math in torch.

    Returns (du, dap, dbp, dkp, dx, dW_in, dW_a, db_a, d_floor,
             dW_b, db_b, dW_k, db_k). Used for local verification only.
    """
    B, T, S = x.shape
    # recompute projections (same as forward)
    u = F.linear(x, w_in)
    ap = F.linear(x, w_a, b_a) + floor
    bp = F.linear(x, w_b, b_b)
    kp = F.linear(x, w_k, b_k)
    kr = keep.unsqueeze(-1)
    alpha = torch.sigmoid(ap)
    beta = torch.sigmoid(bp)
    kk = torch.sigmoid(kp)
    su = torch.sigmoid(u)
    v = u * su
    silu_p = su * (1.0 + u * (1.0 - su))
    k2 = kk * kk
    omb = 1.0 - beta * k2
    du = torch.empty_like(u)
    dap = torch.empty_like(ap)
    dbp = torch.empty_like(bp)
    dkp = torch.empty_like(kp)
    D = torch.zeros(B, S, dtype=torch.float32)
    for ti in range(T):
        t = T - 1 - ti
        a_next = a_seq[:, t + 1] if t + 1 < T else torch.zeros(B, S)
        D = dh[:, t] + a_next * D
        h_prev = h_seq[:, t - 1] if t > 0 else torch.zeros(B, S)
        du[:, t] = D * kr[:, t] * beta[:, t] * kk[:, t] * silu_p[:, t]
        dap[:, t] = (D * h_prev * kr[:, t] * alpha[:, t]
                     * (1.0 - alpha[:, t]) * omb[:, t])
        dbp[:, t] = (D * kr[:, t] * beta[:, t] * (1.0 - beta[:, t])
                     * (kk[:, t] * v[:, t]
                        - alpha[:, t] * k2[:, t] * h_prev))
        dkp[:, t] = (D * kr[:, t] * beta[:, t] * kk[:, t]
                     * (1.0 - kk[:, t])
                     * (v[:, t] - 2.0 * alpha[:, t] * kk[:, t] * h_prev))
    # NOTE: F.linear computes x @ W.T, so dL/dx = dL/du @ W (NOT W.t()).
    # (PyTorch Linear backward: grad_input = grad_output @ weight.)
    dx = du @ w_in + dap @ w_a + dbp @ w_b + dkp @ w_k
    # flat single-GEMM over B*T — PyTorch Linear backward's exact
    # reduction order (per-batch einsum diverged at TF32 precision)
    xr = x.reshape(-1, x.shape[-1])
    dW_in = du.reshape(-1, du.shape[-1]).t() @ xr
    dW_a = dap.reshape(-1, dap.shape[-1]).t() @ xr
    dW_b = dbp.reshape(-1, dbp.shape[-1]).t() @ xr
    dW_k = dkp.reshape(-1, dkp.shape[-1]).t() @ xr
    db_a = dap.sum(dim=(0, 1))
    db_b = dbp.sum(dim=(0, 1))
    db_k = dkp.sum(dim=(0, 1))
    d_floor = dap.sum()
    return du, dap, dbp, dkp, dx, dW_in, dW_a, db_a, d_floor, dW_b, db_b, dW_k, db_k


class _FusedProjScanFn(torch.autograd.Function):
    """Save-x-only Function: projections recomputed, never saved."""

    @staticmethod
    def forward(ctx, x, w_in, w_a, b_a, floor, w_b, b_b, w_k, b_k, keep):
        B, T, S = x.shape
        # projections: cuBLAS, transient — NOT saved
        u = F.linear(x, w_in)
        ap = F.linear(x, w_a, b_a) + floor
        bp = F.linear(x, w_b, b_b)
        kp = F.linear(x, w_k, b_k)
        if x.is_cuda:
            h = torch.empty(B, T, S, device=x.device, dtype=torch.float32)
            a = torch.empty(B, T, S, device=x.device, dtype=torch.float32)
            _delta_scan_fwd_kernel[(B,)](
                u.contiguous(), ap.contiguous(), bp.contiguous(),
                kp.contiguous(), keep.contiguous(), h, a, T,
                BLOCK=BLOCK_S, num_warps=4)
        else:
            kr = keep.unsqueeze(-1)
            alpha = torch.sigmoid(ap)
            beta = torch.sigmoid(bp)
            kk = torch.sigmoid(kp)
            a = kr * alpha * (1.0 - beta * kk * kk)
            h = _delta_scan_cpu(u, ap, bp, kp, keep)
        # saved: x + keep + scan outputs. u/ap/bp/kp deliberately absent.
        ctx.save_for_backward(x, keep, h, a,
                              w_in, w_a, b_a, floor, w_b, b_b, w_k, b_k)
        return h

    @staticmethod
    def backward(ctx, dh):
        (x, keep, h, a, w_in, w_a, b_a, floor,
         w_b, b_b, w_k, b_k) = ctx.saved_tensors
        B, T, S = x.shape
        dh = dh.contiguous()
        if x.is_cuda:
            # recompute projections (cuBLAS, transient)
            u = F.linear(x, w_in)
            ap = F.linear(x, w_a, b_a) + floor
            bp = F.linear(x, w_b, b_b)
            kp = F.linear(x, w_k, b_k)
            hs = torch.zeros(B, T + 1, S, device=x.device,
                             dtype=torch.float32)
            hs[:, 1:] = h                      # hs[t] = h_{t-1}
            apn = torch.zeros(B, T + 1, S, device=x.device,
                              dtype=torch.float32)
            apn[:, :T] = a                     # apn[t+1] = a_{t+1}
            du = torch.empty_like(u)
            dap = torch.empty_like(ap)
            dbp = torch.empty_like(bp)
            dkp = torch.empty_like(kp)
            _delta_scan_bwd_kernel[(B,)](
                dh, hs, apn, u.contiguous(), ap.contiguous(),
                bp.contiguous(), kp.contiguous(), keep.contiguous(),
                du, dap, dbp, dkp, T, BLOCK=BLOCK_S, num_warps=4)
            # NOTE: F.linear computes x @ W.T, so dL/dx = dL/du @ W (NOT W.t()).
            # (PyTorch Linear backward: grad_input = grad_output @ weight.)
            dx = du @ w_in + dap @ w_a + dbp @ w_b + dkp @ w_k
            # flat single-GEMM over B*T — PyTorch Linear backward's exact
            # reduction order (per-batch einsum diverged at TF32 precision)
            xr = x.reshape(-1, x.shape[-1])
            dW_in = du.reshape(-1, du.shape[-1]).t() @ xr
            dW_a = dap.reshape(-1, dap.shape[-1]).t() @ xr
            dW_b = dbp.reshape(-1, dbp.shape[-1]).t() @ xr
            dW_k = dkp.reshape(-1, dkp.shape[-1]).t() @ xr
            db_a = dap.sum(dim=(0, 1))
            db_b = dbp.sum(dim=(0, 1))
            db_k = dkp.sum(dim=(0, 1))
            d_floor = dap.sum()
        else:
            (du, dap, dbp, dkp, dx, dW_in, dW_a, db_a, d_floor,
             dW_b, db_b, dW_k, db_k) = _adjoint_torch(
                dh, x, keep, h, a,
                w_in, w_a, b_a, floor, w_b, b_b, w_k, b_k)
        return (dx, dW_in, dW_a, db_a, d_floor,
                dW_b, db_b, dW_k, db_k, None)


class SelectiveSegmentedStateV8AFused(SelectiveSegmentedStateV8A):
    """V8-A with fused projection+scan (Path B). Same params, same math.

    __init__ inherited unchanged -> identical params under the same seed.
    out_gate_proj stays a separate linear (its output is consumed after
    the scan; fusing it saves nothing).
    """

    def forward(self, x, reset=None):
        if reset is not None:
            keep = (1.0 - reset).float()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = _FusedProjScanFn.apply(
            x, self.in_proj.weight,
            self.gate_proj.weight, self.gate_proj.bias, self.forget_floor,
            self.write_proj.weight, self.write_proj.bias,
            self.key_proj.weight, self.key_proj.bias,
            keep)
        self.last_state = h_seq.detach()  # stability logging only, as V8-A
        og = self.out_gate_proj(x).float()
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


"""(inlined) V12 multimodal helpers.

New code only — models.py / triton_kernels.py / fused_mixer.py are FROZEN
(his V12 discipline). This file adds modality encoders + a continuous
decoder-only AR model reusing the frozen Block.

Task (his call): autoregressive next-element prediction (MSE), the direct
analogue of the text campaign's val-loss comparison.
"""

import math

# (imports stripped)


# ---------------------------------------------------------------- encoders

class VisionEncoder(nn.Module):
    """Patchify + linear: (B,3,H,W) -> (B,T,dim); elements: (B,T,E) raw patches."""

    def __init__(self, dim, patch=8, in_ch=3):
        super().__init__()
        self.patch = patch
        self.in_ch = in_ch
        self.elem_dim = in_ch * patch * patch
        self.proj = nn.Linear(self.elem_dim, dim)

    def elements(self, img):
        B, C, H, W = img.shape
        p = self.patch
        assert H % p == 0 and W % p == 0, (H, W, p)
        x = img.view(B, C, H // p, p, W // p, p)
        x = x.permute(0, 2, 4, 1, 3, 5).contiguous()
        return x.view(B, (H // p) * (W // p), self.elem_dim)

    def forward(self, img):
        return self.proj(self.elements(img))

    @property
    def seq_len(self):
        return None  # set by image size at runtime


class AudioEncoder(nn.Module):
    """Linear on log-mel frames: (B,T,80) -> (B,T,dim); elements = mel itself."""

    def __init__(self, dim, n_mels=80):
        super().__init__()
        self.elem_dim = n_mels
        self.proj = nn.Linear(n_mels, dim)

    def elements(self, mel):
        return mel

    def forward(self, mel):
        return self.proj(mel)


# ------------------------------------------------------- log-mel (no deps)

_mel_fb_cache = {}


def log_mel(wave, sr=16000, n_mels=80, n_fft=400, hop=160, win=400):
    """wave: (B,N) float32 -> (B,T,n_mels) log-mel. Pure torch."""
    key = (sr, n_mels, n_fft)
    if key not in _mel_fb_cache:
        fb = _mel_filterbank(sr, n_fft, n_mels)
        _mel_fb_cache[key] = fb
    else:
        fb = _mel_fb_cache[key]
    fb = fb.to(wave.device, wave.dtype)
    window = torch.hann_window(win, device=wave.device, dtype=wave.dtype)
    spec = torch.stft(wave, n_fft=n_fft, hop_length=hop, win_length=win,
                      window=window, return_complex=True)  # (B,F,Tt)
    power = spec.abs().pow(2).transpose(1, 2)  # (B,Tt,F)
    mel = power @ fb.T  # (B,Tt,n_mels)
    return torch.log(torch.clamp(mel, min=1e-10))


def _mel_filterbank(sr, n_fft, n_mels, fmin=0.0, fmax=None):
    fmax = fmax or sr / 2
    mmin, mmax = _hz_to_mel(fmin), _hz_to_mel(fmax)
    m_pts = torch.linspace(mmin, mmax, n_mels + 2)
    h_pts = _mel_to_hz(m_pts)
    f_pts = torch.floor((n_fft + 1) * h_pts / sr).long()
    fb = torch.zeros(n_mels, n_fft // 2 + 1)
    for m in range(1, n_mels + 1):
        f0, f1, f2 = f_pts[m - 1].item(), f_pts[m].item(), f_pts[m + 1].item()
        if f1 > f0:
            fb[m - 1, f0:f1] = (torch.arange(f0, f1) - f0) / (f1 - f0)
        if f2 > f1:
            fb[m - 1, f1:f2] = (f2 - torch.arange(f1, f2)) / (f2 - f1)
    return fb


def _hz_to_mel(hz):
    return 2595.0 * torch.log10(torch.as_tensor(1.0) + hz / 700.0)


def _mel_to_hz(mel):
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)


# ------------------------------------------------------------------- model

class MultimodalAR(nn.Module):
    """Decoder-only AR model over continuous embeddings.

    encoder: VisionEncoder | AudioEncoder. Causal next-element prediction;
    loss = MSE(pred[:, :-1], elements[:, 1:]).
    """

    def __init__(self, encoder, dim=256, n_layers=8, n_heads=8, seq_len=256,
                 mixer_fn=None, dropout=0.0, ffn_hidden=None):
        super().__init__()
        self.encoder = encoder
        self.elem_dim = encoder.elem_dim
        self.seq_len = seq_len
        self.pos_emb = nn.Embedding(seq_len, dim)
        self.blocks = nn.ModuleList(
            [Block(dim, n_heads, mixer_fn(dim, n_heads), dropout, ffn_hidden)
             for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, self.elem_dim, bias=False)
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, raw):
        B = raw.shape[0]
        T = self.encoder.elements(raw).shape[1]
        x = self.encoder(raw) + self.pos_emb(
            torch.arange(T, device=raw.device))[None]
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln_f(x))  # (B,T,elem_dim)

    def ar_loss(self, raw):
        pred = self(raw)
        with torch.no_grad():
            tgt = self.encoder.elements(raw)
        return F.mse_loss(pred[:, :-1], tgt[:, 1:])


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def match_ffn_hidden(v8a_ffn, dim, mixer_v8a_params, mixer_attn_params):
    """FFN hidden size for the attention arm so totals match (V4-style).

    Per-unit cost of FFN hidden: 2*dim (weights) + 1 (fc1 bias).
    """
    delta = mixer_v8a_params - mixer_attn_params
    per_unit = 2 * dim + 1
    return v8a_ffn + math.ceil(delta / per_unit)


In [ ]:
import time, gc, random, statistics
import torch.nn as nn
import torch.utils.checkpoint as _ckpt

DIM, LAYERS, HEADS, FFN_V8A = 256, 8, 8, 766
T_TXT, PATCH, IMG, BS = 32, 8, 128, 32
T_VIS = (IMG // PATCH) ** 2  # 256
T_JOINT = T_TXT + T_VIS  # 288

_orig_block_forward = Block.forward
def _block_forward_ckpt(self, x, reset=None):
    x = x + self.mix(self.ln1(x), reset)
    x = x + _ckpt.checkpoint(self.mlp, self.ln2(x), use_reentrant=False)
    return x
Block.forward = _block_forward_ckpt

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

# ---- Vision-only model (arms A/B): same as Phase 1, Flickr8k data ----
class VisAR(nn.Module):
    def __init__(self, dim, n_layers, n_heads, seq_len, mixer_fn, ffn_hidden):
        super().__init__()
        self.encoder = VisionEncoder(dim, patch=PATCH)
        self.pos_emb = nn.Embedding(seq_len, dim)
        self.blocks = nn.ModuleList(
            [Block(dim, n_heads, mixer_fn(dim, n_heads), 0.0, ffn_hidden)
             for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, 3*PATCH*PATCH, bias=False)
        self.apply(self._init)
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
    def forward(self, img):
        x = self.encoder(img)
        x = x + self.pos_emb(torch.arange(x.shape[1], device=x.device))[None]
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln_f(x))
    def loss(self, img):
        pred = self(img)
        with torch.no_grad():
            tgt = self.encoder.elements(img)
        # IDENTICAL to joint vision loss: MSE(pred[:, :-1], tgt[:, 1:])
        return F.mse_loss(pred[:, :-1], tgt[:, 1:])

# ---- Joint model (arms C/D): same as pilot ----
class JointEncoder(nn.Module):
    def __init__(self, dim, vocab, patch=8):
        super().__init__()
        self.txt_emb = nn.Embedding(vocab, dim)
        self.patch = patch
        self.elem_dim_vis = 3 * patch * patch
        self.proj_vis = nn.Linear(self.elem_dim_vis, dim)
    def elements_vis(self, img):
        B, C, H, W = img.shape
        p = self.patch
        x = img.unfold(2, p, p).unfold(3, p, p)
        x = x.permute(0, 2, 3, 1, 4, 5).contiguous()
        return x.view(B, -1, self.elem_dim_vis)
    def forward(self, img, cap_ids):
        t = self.txt_emb(cap_ids)
        v = self.proj_vis(self.elements_vis(img))
        return torch.cat([t, v], dim=1)

class JointAR(nn.Module):
    def __init__(self, dim, n_layers, n_heads, seq_len, mixer_fn, ffn_hidden):
        super().__init__()
        self.encoder = JointEncoder(dim, VOCAB, PATCH)
        self.pos_emb = nn.Embedding(seq_len, dim)
        self.blocks = nn.ModuleList(
            [Block(dim, n_heads, mixer_fn(dim, n_heads), 0.0, ffn_hidden)
             for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head_vis = nn.Linear(dim, 3*PATCH*PATCH, bias=False)
        self.head_txt = nn.Linear(dim, dim, bias=False)
        self.apply(self._init)
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
    def forward(self, img, cap_ids):
        x = self.encoder(img, cap_ids)
        x = x + self.pos_emb(torch.arange(T_JOINT, device=x.device))[None]
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        pv = self.head_vis(x[:, T_TXT:T_JOINT-1])
        pt = self.head_txt(x[:, :T_TXT-1])
        return pv, pt
    def vis_loss(self, img, cap_ids):
        # IDENTICAL definition: MSE on patch predictions
        pv, _ = self(img, cap_ids)
        with torch.no_grad():
            tv = self.encoder.elements_vis(img)[:, 1:]
        return F.mse_loss(pv, tv)

def build_vis(mixer, seed, ffn_hidden):
    set_seed(seed)
    m = VisAR(DIM, LAYERS, HEADS, T_VIS, mixer, ffn_hidden).to(device)
    import math as _math
    _floors = torch.linspace(_math.log(0.3/0.7), _math.log(0.9/0.1), LAYERS)
    for _i, _blk in enumerate(m.blocks):
        if hasattr(_blk.mix, "forget_floor"):
            _blk.mix.forget_floor.data.fill_(_floors[_i])
    return m, torch.optim.AdamW(m.parameters(), lr=3e-4)

def build_joint(mixer, seed, ffn_hidden):
    set_seed(seed)
    m = JointAR(DIM, LAYERS, HEADS, T_JOINT, mixer, ffn_hidden).to(device)
    import math as _math
    _floors = torch.linspace(_math.log(0.3/0.7), _math.log(0.9/0.1), LAYERS)
    for _i, _blk in enumerate(m.blocks):
        if hasattr(_blk.mix, "forget_floor"):
            _blk.mix.forget_floor.data.fill_(_floors[_i])
    return m, torch.optim.AdamW(m.parameters(), lr=3e-4)

# param matching
torch.manual_seed(0)
_m0 = VisAR(DIM, 1, HEADS, T_VIS,
            lambda d,h: SelectiveSegmentedStateV8AFused(d, state_dim=256),
            ffn_hidden=FFN_V8A)
_pv = sum(p.numel() for p in _m0.blocks[0].mix.parameters())
_ma = VisAR(DIM, 1, HEADS, T_VIS,
            lambda d,h: CausalSelfAttention(d, h), ffn_hidden=FFN_V8A)
_pa = sum(p.numel() for p in _ma.blocks[0].mix.parameters())
FFN_ATTN = match_ffn_hidden(FFN_V8A, DIM, _pv, _pa)
del _m0, _ma
print(f"mixer params/layer: v8a {_pv} vs attn {_pa} -> ffn {FFN_V8A} vs {FFN_ATTN}")

# FIXED data (seed 42 train, 43 val) — same for all four arms
frng = random.Random(42)
idx_tr = [frng.randrange(len(items)) for _ in range(BS)]
vrng = random.Random(43)
idx_va = [vrng.randrange(len(items)) for _ in range(BS)]
def load_pair(i):
    name, caption = items[i]
    im = load_image(name).resize((IMG, IMG), Image.BILINEAR)
    t = torch.from_numpy(__import__("numpy").array(im)).permute(2,0,1).float()/255.0
    cap = encode_caption(caption, T_TXT)
    return t, torch.tensor(cap, dtype=torch.long)
raw_tr = [load_pair(i) for i in idx_tr]
raw_va = [load_pair(i) for i in idx_va]
img_tr = ((torch.stack([r[0] for r in raw_tr]) - IMAGENET_MEAN) / IMAGENET_STD).to(device)
cap_tr = torch.stack([r[1] for r in raw_tr]).to(device)
img_va = ((torch.stack([r[0] for r in raw_va]) - IMAGENET_MEAN) / IMAGENET_STD).to(device)
cap_va = torch.stack([r[1] for r in raw_va]).to(device)
print(f"train: img {tuple(img_tr.shape)} cap {tuple(cap_tr.shape)}")
print(f"val:   img {tuple(img_va.shape)} cap {tuple(cap_va.shape)}")

# INIT VERIFICATION (his requirement): tensor-level equality on shared comps
def verify_init(m1, m2, label):
    # compare encoder + pos_emb (shared architecture components)
    for (n1, p1), (n2, p2) in zip(m1.named_parameters(), m2.named_parameters()):
        if "mix" in n1 or "mix" in n2:
            continue  # mixer differs by design
        if n1 != n2:
            continue  # different names (e.g., head vs head_vis)
        if not torch.equal(p1.cpu(), p2.cpu()):
            print(f"INIT MISMATCH {label}: {n1}", flush=True)
            return False
    print(f"INIT OK {label}: shared components match", flush=True)
    return True

# Build all four arms
print("Building 4 arms...", flush=True)
mA, oA = build_vis(lambda d,h: CausalSelfAttention(d,h), 0, FFN_ATTN)      # A: vis/attn
mB, oB = build_vis(lambda d,h: SelectiveSegmentedStateV8AFused(d, state_dim=256), 0, FFN_V8A)  # B: vis/v8a
mC, oC = build_joint(lambda d,h: CausalSelfAttention(d,h), 0, FFN_ATTN)    # C: joint/attn
mD, oD = build_joint(lambda d,h: SelectiveSegmentedStateV8AFused(d, state_dim=256), 0, FFN_V8A)  # D: joint/v8a

verify_init(mA, mB, "A/B vision encoders")
verify_init(mC, mD, "C/D joint encoders")

def train_vis(m, o, steps=500):
    m.train()
    for _ in range(5):  # warmup
        o.zero_grad(set_to_none=True)
        m.loss(img_tr).backward(); o.step()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(steps):
        o.zero_grad(set_to_none=True)
        m.loss(img_tr).backward(); o.step()
    torch.cuda.synchronize()
    ms = 1000*(time.perf_counter()-t0)/steps
    m.eval()
    with torch.no_grad():
        lv = m.loss(img_va)
    return lv.item(), ms

def train_joint(m, o, steps=500):
    m.train()
    for _ in range(5):
        o.zero_grad(set_to_none=True)
        tot, _, _ = m.loss(img_tr, cap_tr)
        tot.backward(); o.step()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(steps):
        o.zero_grad(set_to_none=True)
        tot, _, _ = m.loss(img_tr, cap_tr)
        tot.backward(); o.step()
    torch.cuda.synchronize()
    ms = 1000*(time.perf_counter()-t0)/steps
    m.eval()
    with torch.no_grad():
        lv = m.vis_loss(img_va, cap_va)
    return lv.item(), ms

# JointAR needs loss() returning (total, lv, lt) for training; add it
_orig_joint_loss = JointAR.vis_loss
def _joint_total_loss(self, img, cap_ids):
    pv, pt = self(img, cap_ids)
    with torch.no_grad():
        tv = self.encoder.elements_vis(img)[:, 1:]
        tt = self.encoder.txt_emb(cap_ids)[:, 1:].detach()
    lv = F.mse_loss(pv, tv)
    lt = F.mse_loss(pt, tt)
    return (lv+lt)/2, lv.detach(), lt.detach()
JointAR.loss = _joint_total_loss

results = {}
print("Training A (vis/attn)...", flush=True)
torch.cuda.reset_peak_memory_stats()
lvA, msA = train_vis(mA, oA)
paA = torch.cuda.max_memory_allocated()/1e6; prA = torch.cuda.max_memory_reserved()/1e6
results["A"] = (lvA, msA, paA, prA)
del mA, oA; gc.collect(); torch.cuda.empty_cache()

print("Training B (vis/v8a)...", flush=True)
torch.cuda.reset_peak_memory_stats()
lvB, msB = train_vis(mB, oB)
paB = torch.cuda.max_memory_allocated()/1e6; prB = torch.cuda.max_memory_reserved()/1e6
results["B"] = (lvB, msB, paB, prB)
del mB, oB; gc.collect(); torch.cuda.empty_cache()

print("Training C (joint/attn)...", flush=True)
torch.cuda.reset_peak_memory_stats()
lvC, msC = train_joint(mC, oC)
paC = torch.cuda.max_memory_allocated()/1e6; prC = torch.cuda.max_memory_reserved()/1e6
results["C"] = (lvC, msC, paC, prC)
del mC, oC; gc.collect(); torch.cuda.empty_cache()

print("Training D (joint/v8a)...", flush=True)
torch.cuda.reset_peak_memory_stats()
lvD, msD = train_joint(mD, oD)
paD = torch.cuda.max_memory_allocated()/1e6; prD = torch.cuda.max_memory_reserved()/1e6
results["D"] = (lvD, msD, paD, prD)
del mD, oD; gc.collect(); torch.cuda.empty_cache()

# Predeclared interaction
d_vis = results["B"][0] - results["A"][0]    # Δ vision-only
d_joint = results["D"][0] - results["C"][0]  # Δ joint
I = d_joint - d_vis

lines = []
lines.append(f"A vis/attn:  val_vis {results['A'][0]:.4f} | {results['A'][1]:.1f} ms/step alloc {results['A'][2]:.0f} res {results['A'][3]:.0f}")
lines.append(f"B vis/v8a:   val_vis {results['B'][0]:.4f} | {results['B'][1]:.1f} ms/step alloc {results['B'][2]:.0f} res {results['B'][3]:.0f}")
lines.append(f"C joint/attn: val_vis {results['C'][0]:.4f} | {results['C'][1]:.1f} ms/step alloc {results['C'][2]:.0f} res {results['C'][3]:.0f}")
lines.append(f"D joint/v8a:  val_vis {results['D'][0]:.4f} | {results['D'][1]:.1f} ms/step alloc {results['D'][2]:.0f} res {results['D'][3]:.0f}")
lines.append(f"Δ_vis-only = {d_vis:.4f} | Δ_joint = {d_joint:.4f} | I = {I:.4f}")
lines.append("I<0: rescue | I≈0: no interaction | I>0: joint hurts V8-A")
for L in lines:
    print(L, flush=True)
with open("/content/v12p4_2x2.txt", "w") as f:
    f.write("\n".join(lines) + "\n")
from google.colab import files
files.download("/content/v12p4_2x2.txt")
print("SAVE 2x2 DONE")
